In [1]:
import json
import os
import pandas as pd
import pickle

# Data Extraction and Summarization

- subject data are in 'subId.json' files
- read each file and summarizes the important bird task trial information for further analysis

*** specify the subset data source ***

In [2]:
# specify the folder name containing the subset of data collected
label = ""


In [3]:
def read_json_files(folder_path):
    """
    Reads all JSON files in a given folder and returns a dictionary
    containing file names as keys and parsed JSON data as values.
    
    Args:
    - folder_path (str): Path to the folder containing JSON files.
    
    Returns:
    - dict: A dictionary containing file names as keys and parsed JSON data as values.
    """
    json_data = {}
    for filename in os.listdir(folder_path):
        if filename.endswith('.json'):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r') as file:
                json_data[filename] = json.load(file)
    return json_data
    

In [17]:
def summarize_save_data(json_data, label):
    """
    Summarizes the structure of JSON data.
    And save the extracted bird task data to csv.
    
    Args:
    - json_data (dict): A dictionary containing JSON data.
    - label (str): folder name containing the subset of data collected. 
    """
    dfs_trials = pd.DataFrame() # Dataframes containing bird task trials
    saved_participants = [] # Summary of saved participant data

    # iterate over all the participants .json files
    for filename, data in json_data.items():
        # summarize index dictionary for important trials
        print(f"Summary for file: {filename}")
        data_summary = _summarize(data)
        
        # for all subjects extract and save important data 
        trials_dict = extract_data(data, data_summary) 
        subject_id = filename.rsplit('.', 1)[0]
        df_trials = pd.DataFrame(trials_dict, index = [subject_id]*200)
        saved_participants.append(subject_id)

        dfs_trials = pd.concat([dfs_trials, df_trials], axis = 0)

    dfs_trials = dfs_trials.rename_axis('subId')
    # save trials data and survey data to csv
    # create folder if not exist
    if not os.path.exists('../processed_data'):
        os.makedirs('../processed_data')
    output_file = '../processed_data/' + label + "_trials.csv"
    dfs_trials.to_csv(output_file)
    print(f"---FILE SAVED ({len(saved_participants)})---") # report the number of participants saved

    # save list of participant ID
    output_pkl = '../processed_data/' + label + '_subID.pkl'
    with open(output_pkl, 'wb') as file:
        pickle.dump(saved_participants, file)


In [21]:
def _summarize(data):
    """
    Helper function to summarize JSON data structure.
    
    Args:
    - data: JSON data (dict) to be summarized.
    
    Returns:
    - dict: A dictionary containing labeled task index.
    """
    print(f"{' '*(4)}Data length: ", len(data))
    
    view_history = [] # view_history
    view_history_idx = []
    comprehensions = [] # responses & num_errors
    comprehensions_idx = []
    trials = [] # bird_position (blocks)
    trials_idx = []
    others = [] # others
    others_idx = []

    # use dict keys to identify trials
    for item in range(len(data)):
        # 9 before trials (4 more if failed comprehension) + 10 after trials (more if failed attention check) = 19 total
        if 'view_history' in data[item].keys():
            view_history.append(data[item])
            view_history_idx.append(item)
        # 2 comprehensions (3 if failed once)
        elif 'responses' and 'num_errors' in data[item].keys(): 
            comprehensions.append(data[item])
            comprehensions_idx.append(item)
        # 23 + 200 (50 trials * 4 blocks) = 223 
        elif 'bird_position' in data[item].keys():
            trials.append(data[item])
            trials_idx.append(item)
        # screen checks # 10 screen checks
        else: 
            others.append(data[item])
            others_idx.append(item)
            
    # detailed labels (rpm and blocks)
    # the last 200 trials contains 4*50 bird task trial data
    blocks = trials[-200:]
    blocks_idx = trials_idx[-200:]

    # output summary data index
    data_summary = {'comprehensions_idx': comprehensions_idx,
                    'blocks_idx': blocks_idx,
                   } 
    
    print('Number of view history pages: ', len(view_history_idx))
    print('Number of comprehension pages: ', len(comprehensions_idx))
    print('Number of trials (including instruction demo): ', len(trials_idx))
    print('Number of actual trials: ', len(blocks_idx))
    print('Number of other pages: ', len(others_idx))  
    total = len(view_history_idx) + + len(comprehensions_idx) + len(trials_idx) + len(others_idx)
    print('Total recorded pages: ', total)
    
    return data_summary

In [22]:
def extract_data(data, data_summary):
    """
    Extract needed data.
    params extracted: 'trials': ['bird_position', 'bag_position', 'bucket_position', 'completed', 'stayed', 'block', 'randomized', 'trial', 'time_elapsed']
    
    Args:
    - data: JSON data (dict) to be summarized.
    - data_summary: dict containing summarized index with labels
    
    Returns:
    - dict: dictionary containing extracted data with labels.
    """
    # Extract important bird task trial data
    keys_to_extract = ['trial', 'block', 'randomized', 
                       'bird_position', 'bag_position', 'bucket_position', 
                       'completed', 'stayed', 'time_elapsed']
    blocks_idx = data_summary['blocks_idx']
    trials_dict = []
    for t in blocks_idx:
        trial = data[t]
        # Extract dictionary using dictionary comprehension
        extracted_dict = {key: trial[key] for key in keys_to_extract}
        trials_dict.append(extracted_dict)
#     # quality check 
#     a = 0
#     b = 0
#     c = 0
#     d = 0
#     for t in trials_dict:
#         if t['block'] == 1:
#             a+=1
#         if t['block'] == 2:
#             b+=1
#         if t['block'] == 3:
#             c+=1    
#         if t['block'] == 4:
#             d+=1 
#     print(a,b,c,d)

    return trials_dict

In [23]:
# Example usage
folder_path = "../data/"
os.chdir(folder_path)

json_data = read_json_files(folder_path + label)
summarize_save_data(json_data, label)

Summary for file: sample_subId.json
    Data length:  243
Number of view history pages:  10
Number of comprehension pages:  2
Number of trials (including instruction demo):  223
Number of actual trials:  200
Number of other pages:  8
Total recorded pages:  243
---FILE SAVED (1)---
